# Segmentation Evaluation Notebook: YOLOv11 + SAM2 [Night Split]

## Step 0A. Packages installation and environment setup

In [ ]:
# Step 0A: Install required packages for scientific segmentation evaluation
import sys
import subprocess
import importlib

# Defining a function to check the package installation
def install_if_missing(package_name, import_name=None, extra_pip_args=None):
    import_name = import_name or package_name
    extra_pip_args = extra_pip_args or []
    try:
        importlib.import_module(import_name) 
        print(f"{package_name} already installed") 
        return True
    except ImportError:
        print(f"[INSTALLING] {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, *extra_pip_args])
            importlib.import_module(import_name)
            print(f"{package_name} installed successfully")
            return True
        except Exception as e:
            print(f"[WARNING] Could not install {package_name}: {e}")
            return False

# Printing
install_if_missing("numpy") 
install_if_missing("pandas") 
install_if_missing("matplotlib") 
install_if_missing("opencv-python", "cv2") 
install_if_missing("Pillow", "PIL") 
install_if_missing("tqdm") 
install_if_missing("pycocotools") 
install_if_missing("ultralytics") 
install_if_missing("scipy") 
install_if_missing("psutil") 
install_if_missing("hydra-core", "hydra") 
install_if_missing("omegaconf") 
install_if_missing("iopath") 

try:
    install_if_missing("GPUtil") 
except Exception as e:
    print(f"[INFO] GPUtil installation skipped or failed: {e}") 

try:
    import sam2  
    print("SAM2 package is available.")
except Exception as sam2_import_error:
    print("SAM2 package is not currently importable in this environment.")
    print(f"SAM2 import error: {repr(sam2_import_error)}")

## Step 0B. Libraries and imports

In [ ]:
# Step 0B: Libraries for file handling, computation, model loading, and visualization
from pathlib import Path
import os
import json
import math
import time
import random
import platform
import traceback
import shutil
import re
import ast
import copy
import psutil
import torch
import cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from PIL import Image
from tqdm import tqdm
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from matplotlib.ticker import AutoMinorLocator
from collections import defaultdict, Counter

try:
    import GPUtil
except Exception:
    GPUtil = None

try:
    from IPython.display import display
except Exception:
    display = print

## Step 0C. Reproducibility settings

In [ ]:
# Step 0C: Reproducibility configuration for deterministic scientific evaluation

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available(): 
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Print
print(f"Global random seed set to: {SEED}")

## Step 0D. Machine and runtime specifications

In [ ]:
# Step 0D: Machine and environment specification logging for scientific reporting
def get_machine_specs():
    specs = {
        "platform_system": platform.system(), 
        "platform_release": platform.release(), 
        "platform_version": platform.version(), 
        "architecture": platform.machine(), 
        "processor": platform.processor(), 
        "python_version": platform.python_version(), 
        "physical_cpu_cores": psutil.cpu_count(logical=False), 
        "logical_cpu_cores": psutil.cpu_count(logical=True), 
        "total_ram_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2), 
        "available_ram_gb": round(psutil.virtual_memory().available / (1024 ** 3), 2), 
        "torch_version": torch.__version__, 
        "cuda_available": torch.cuda.is_available(), 
        "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0, 
    }
# Function for CUDA availability
    if torch.cuda.is_available():
        specs["gpu_name_torch"] = torch.cuda.get_device_name(0) 
        gpu_props = torch.cuda.get_device_properties(0) 
        specs["gpu_total_memory_gb"] = round(gpu_props.total_memory / (1024 ** 3), 2) 

    if GPUtil is not None:
        try:
            gpus = GPUtil.getGPUs()
            if len(gpus) > 0:
                specs["gpu_name_gputil"] = gpus[0].name
                specs["gpu_driver_version"] = getattr(gpus[0], "driver", "NA")
                specs["gpu_memory_total_mb"] = getattr(gpus[0], "memoryTotal", "NA")
        except Exception as e:
            specs["gpu_query_warning"] = repr(e)

    return specs

MACHINE_SPECS = get_machine_specs()
machine_specs_df = pd.DataFrame([MACHINE_SPECS]) 

print("Machine or laptop, and runtime specifications have been collected.")
display(machine_specs_df) 

## Step 1A. Main project paths and configuration

In [ ]:
# Step 1A: Project path and configuration 
# Reader's notice: Behavioral classes categories: 0 = cow [only reference category]
# 1 = Drinking; 2 = Feeding; 3 = Resting; 4 = Standing; and 5 = Walking

# Verified base root for dataset
DEFAULT_WINDOWS_BASE_ROOT = Path(r"C:\\\\") 
ENV_BASE_ROOT = os.environ.get("", "").strip() 

if ENV_BASE_ROOT:
    BASE_ROOT = Path(ENV_BASE_ROOT)
elif DEFAULT_WINDOWS_BASE_ROOT.exists():
    BASE_ROOT = DEFAULT_WINDOWS_BASE_ROOT
else:
    BASE_ROOT = Path.cwd() 

# Verified dataset and project folders
COCO_ROOT = BASE_ROOT / "" 
COCO_SPLIT_DIR = COCO_ROOT / "train" 
COCO_JSON_PATH = COCO_SPLIT_DIR / "" 

DAY_MASK_ROOT = BASE_ROOT / "" 
NIGHT_MASK_ROOT = BASE_ROOT / "" 

MODELS_ROOT = BASE_ROOT / "models" 
RUNS_ROOT = BASE_ROOT / "runs" 
RUNPOD_ROOT = Path(r"C:\\\\") 

# SAM2 model configuration and paths
MODEL_NAME = ""
SAM2_CHECKPOINT_PATH = MODELS_ROOT / "sam2_hiera_small.pt" 
YOLO_FINAL_WEIGHTS_PATH = RUNPOD_ROOT / "" / "run" / "weights" / "best.pt" 
RTDETR_WEIGHTS_PATH = RUNS_ROOT / "" / "weights" / "best.pt" 

# COCO Segmentation Evaluation split
EVAL_SPLIT_NAME = "night" 
NIGHT_FILTER_TOKEN = "_night_" 

# Behavioral evaluation categories
COW_CATEGORY_ID = 0  
COCO_EVAL_CATEGORY_ID = 1  
VALID_COW_GT_CATEGORY_IDS = [1, 2, 3, 4, 5] 
BEHAVIOR_CATEGORY_IDS = [1, 2, 3, 4, 5] 

# Preserving the validated working prediction path:
USE_CLASS_FILTER = False
TARGET_CLASS_ID = None

# Detector settings
DETECTOR_NAME = "" 
DETECTOR_WEIGHTS_PATH = YOLO_FINAL_WEIGHTS_PATH 
# Detector configuration
CONF_THRESHOLD = 0.40 
IOU_THRESHOLD = 0.50 
IMAGE_SIZE_FOR_DETECTOR = 768 
MAX_DETECTIONS_PER_IMAGE = 15 

# Device/laptop settings
CUDA_AVAILABLE = torch.cuda.is_available() 
SAM2_DEVICE = "cuda" if CUDA_AVAILABLE else "cpu" 
YOLO_DEVICE = 0 if CUDA_AVAILABLE else "cpu" 

# Evaluation metrics setting
PRIMARY_INSTANCE_IOU_MATCH_THRESHOLD = 0.50 
IOU_ANALYSIS_THRESHOLDS = np.round(np.arange(0.50, 1.00, 0.05), 2) 

# Optional setting for seed dataset to check the analysis speed
SAVE_MASK_IMAGES = False
SAVE_OVERLAY_IMAGES = False
SAVE_ONLY_FIRST_N_OVERLAYS = 0
SAVE_WORST_CASE_OVERLAYS = 0

# Difficulty grouping by ground truth cow count
EASY_MAX_COWS = 2 
MEDIUM_MAX_COWS = 5 

# Output folder structure
EVAL_OUTPUT_ROOT = RUNPOD_ROOT / "" / MODEL_NAME / "" 
PRED_MASK_DIR = EVAL_OUTPUT_ROOT / "" 
GT_MASK_DIR = EVAL_OUTPUT_ROOT / "" 
OVERLAY_DIR = EVAL_OUTPUT_ROOT / "" 
WORST_CASE_DIR = EVAL_OUTPUT_ROOT / "" 
FIG_DIR = EVAL_OUTPUT_ROOT / "" 
CSV_DIR = EVAL_OUTPUT_ROOT / "" 
JSON_DIR = EVAL_OUTPUT_ROOT / "" 
ENV_DIR = EVAL_OUTPUT_ROOT / "" 

# Defining a For Loop for Output Folders
for folder in [
    EVAL_OUTPUT_ROOT, PRED_MASK_DIR, GT_MASK_DIR, OVERLAY_DIR, WORST_CASE_DIR,
    FIG_DIR, CSV_DIR, JSON_DIR, ENV_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Printing
print("Main configurations for the project have been prepared.")
print("BASE_ROOT:", BASE_ROOT) 
print("COCO_JSON_PATH:", COCO_JSON_PATH) 
print("SAM2_CHECKPOINT_PATH:", SAM2_CHECKPOINT_PATH) 
print("YOLO_FINAL_WEIGHTS_PATH:", YOLO_FINAL_WEIGHTS_PATH) 
print("RTDETR_WEIGHTS_PATH:", RTDETR_WEIGHTS_PATH) 
print("EVAL_OUTPUT_ROOT:", EVAL_OUTPUT_ROOT) 
print("Reference cow category ID:", COW_CATEGORY_ID) 
print("Temporary COCO evaluation category ID:", COCO_EVAL_CATEGORY_ID) 
print("Valid ground truth [GT] cow category IDs:", VALID_COW_GT_CATEGORY_IDS) 
print("Behavioral category IDs:", BEHAVIOR_CATEGORY_IDS) 
print("Detector class filtering:", USE_CLASS_FILTER) 
print("CUDA available:", CUDA_AVAILABLE) 
print("SAM2 device:", SAM2_DEVICE) 
print("YOLO device:", YOLO_DEVICE) 
print("Detector image size:", IMAGE_SIZE_FOR_DETECTOR) 
print("Maximum detections per image:", MAX_DETECTIONS_PER_IMAGE) 

## Step 1B:  Path verification

In [ ]:
# Step 1B: Verify all main input paths for the project

required_paths = {
    "BASE_ROOT": BASE_ROOT, 
    "COCO_ROOT": COCO_ROOT, 
    "COCO_SPLIT_DIR": COCO_SPLIT_DIR, 
    "COCO_JSON_PATH": COCO_JSON_PATH, 
    "MODELS_ROOT": MODELS_ROOT, 
    "SAM2_CHECKPOINT_PATH": SAM2_CHECKPOINT_PATH, 
    "RUNPOD_ROOT": RUNPOD_ROOT, 
    "YOLO_FINAL_WEIGHTS_PATH": YOLO_FINAL_WEIGHTS_PATH, 
    "DAY_MASK_ROOT": DAY_MASK_ROOT, 
    "NIGHT_MASK_ROOT": NIGHT_MASK_ROOT, 
}

for name, path_obj in required_paths.items():
    if not path_obj.exists():
        raise FileNotFoundError(f"{name} not found: {path_obj}")
    print(f"{name}: {path_obj}")

# Print
print("\nAll required input paths have been verified successfully.")

## Step 1C: Machine or Laptop and Environment specification reports

In [ ]:
# Step 1C: Machine and environment configuration reports
ENVIRONMENT_JSON = ENV_DIR / "" 
ENVIRONMENT_CSV = ENV_DIR / "" 
CONFIG_JSON = ENV_DIR / "" 

with open(ENVIRONMENT_JSON, "w", encoding="utf-8") as f:
    json.dump(MACHINE_SPECS, f, indent=4)

machine_specs_df.to_csv(ENVIRONMENT_CSV, index=False) 

# Notebook configuration
NOTEBOOK_CONFIG = {
    "seed": SEED, 
    "model_name": MODEL_NAME, 
    "base_root": str(BASE_ROOT), 
    "coco_root": str(COCO_ROOT), 
    "coco_split_dir": str(COCO_SPLIT_DIR), 
    "coco_json_path": str(COCO_JSON_PATH), 
    "sam2_checkpoint_path": str(SAM2_CHECKPOINT_PATH), 
    "detector_weights_path": str(DETECTOR_WEIGHTS_PATH), 
    "detector_name": DETECTOR_NAME, 
    "evaluation_split": EVAL_SPLIT_NAME, 
    "conf_threshold": CONF_THRESHOLD, 
    "iou_threshold": IOU_THRESHOLD, 
    "image_size_for_detector": IMAGE_SIZE_FOR_DETECTOR, 
    "primary_instance_iou_match_threshold": PRIMARY_INSTANCE_IOU_MATCH_THRESHOLD, 
    "ap_thresholds": IOU_ANALYSIS_THRESHOLDS.tolist(), 
    "cow_category_id": COW_CATEGORY_ID, 
    "behavior_category_ids": BEHAVIOR_CATEGORY_IDS, 
    "use_class_filter": USE_CLASS_FILTER, 
    "difficulty_definition": "easy=1-2 cows, medium=3-5 cows, hard=6+ cows",
    "output_root": str(EVAL_OUTPUT_ROOT)
}

with open(CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(NOTEBOOK_CONFIG, f, indent=4)

print("Environment JSON saved:", ENVIRONMENT_JSON)
print("Environment CSV saved:", ENVIRONMENT_CSV)
print("Configuration JSON saved:", CONFIG_JSON)

## Step 2A: COCO dataset annotations

In [ ]:
# Step 2A: Load COCO JSON annotations and inspect dataset size

with open(COCO_JSON_PATH, "r", encoding="utf-8") as f:
    coco_json = json.load(f)

images = coco_json.get("images", [])
annotations = coco_json.get("annotations", [])
categories = coco_json.get("categories", [])

if len(images) == 0:
    raise ValueError("No images found in COCO JSON")
if len(annotations) == 0:
    raise ValueError("No annotations found in COCO JSON")

# Print
print("Total images:", len(images)) 
print("Total annotations:", len(annotations)) 
print("Total categories:", len(categories)) 

categories_df = pd.DataFrame(categories) 
display(categories_df)

## Step 2B: Dataset integrity checks and lookup tables

In [ ]:
# Step 2B: Building annotation lookup tables and running dataset integrity checks
anns_by_image = defaultdict(list)
for ann in annotations:
    anns_by_image[ann["image_id"]].append(ann)

image_id_to_record = {img["id"]: img for img in images}
category_id_to_name = {cat["id"]: cat["name"] for cat in categories}

image_ids = [img["id"] for img in images] 
annotation_ids = [ann["id"] for ann in annotations] 
annotation_image_ids = [ann["image_id"] for ann in annotations] 

duplicate_image_ids = [k for k, v in Counter(image_ids).items() if v > 1] 
duplicate_annotation_ids = [k for k, v in Counter(annotation_ids).items() if v > 1] 
missing_image_refs = sorted(set(annotation_image_ids) - set(image_ids)) 

# Integrity reports
integrity_report = {
    "n_total_images": len(images), 
    "n_total_annotations": len(annotations), 
    "n_categories": len(categories), 
    "n_duplicate_image_ids": len(duplicate_image_ids), 
    "n_duplicate_annotation_ids": len(duplicate_annotation_ids), 
    "n_missing_annotation_image_refs": len(missing_image_refs), 
}

print("Dataset integrity summary.")
for k, v in integrity_report.items():
    print(f"{k}: {v}")

if len(duplicate_image_ids) > 0:
    print("Duplicate image IDs found:", duplicate_image_ids[:20]) 

if len(duplicate_annotation_ids) > 0:
    print("Duplicate annotation IDs found:", duplicate_annotation_ids[:20])

if len(missing_image_refs) > 0:
    print("Annotation image references missing from image table:", missing_image_refs[:20])

INTEGRITY_JSON = JSON_DIR / "dataset_integrity_report.json"
with open(INTEGRITY_JSON, "w", encoding="utf-8") as f:
    json.dump(integrity_report, f, indent=4)

print("Integrity report saved:", INTEGRITY_JSON)

## Step 2C: Helper functions for image naming, path resolution, and metadata

In [ ]:
# Step 2C: Helper function
def get_clean_name(img_record):
    extra = img_record.get("extra", {})
    if isinstance(extra, dict) and extra.get("name"):
        return extra["name"]
    return img_record.get("file_name", "")

def resolve_image_path(img_record):
    clean_name = get_clean_name(img_record)
    file_name = img_record.get("file_name", "")

    candidate_strings = []
    for name in [clean_name, file_name]:
        if name:
            candidate_strings.append(name)
            candidate_strings.append(Path(name).name)

    search_roots = [
        COCO_SPLIT_DIR,
        COCO_ROOT,
        NIGHT_MASK_ROOT,
        DAY_MASK_ROOT,
    ]

    candidates = []
    for root in search_roots:
        for name in candidate_strings:
            candidates.append(root / name)

    seen = set()
    unique_candidates = []
    for p in candidates:
        p = Path(p)
        if str(p) not in seen:
            seen.add(str(p))
            unique_candidates.append(p)

    for p in unique_candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not resolve image path for image ID={img_record.get('id')} | "
        f"clean_name={clean_name} | file_name={file_name}"
    )

def is_night_image(img_record):
    return NIGHT_FILTER_TOKEN in get_clean_name(img_record).lower()

def infer_video_id(file_name):
    stem = Path(file_name).stem
    parts = stem.split("_")
    if len(parts) >= 2 and parts[0].lower() == "video":
        return f"{parts[0]}_{parts[1]}"
    return ""

# Camera/source parsing
def extract_camera(file_name):
    stem = Path(file_name).stem.lower()    
    
    m = re.search(r"camera[_ -]?(\d+)|cam[_ -]?(\d+)|c(\d+)", stem)
    if m:
        camera_num = next(g for g in m.groups() if g is not None)
        return f"Camera {camera_num}"

    return "unknown_camera"

def extract_source_video(file_name):
    vid = infer_video_id(file_name)
    return vid if vid else "unknown_video"

def infer_difficulty_bucket(gt_count):
    if gt_count <= EASY_MAX_COWS:
        return "easy"
    elif gt_count <= MEDIUM_MAX_COWS:
        return "medium"
    return "hard"

## Step 2D: Night-image filtering and dataset summary

In [ ]:
# Step 2D: Filtering night images, verifying file resolution, and summarizing the dataset

night_images = [img for img in images if is_night_image(img)]

if len(night_images) == 0:
    raise ValueError("No night images were found after filtering.")

print("Number of night images selected:", len(night_images))

resolved_rows = []
unresolved_rows = []

for img_record in night_images:
    try:
        img_path = resolve_image_path(img_record)
        resolved_rows.append({ 
            "image_id": img_record["id"], 
            "clean_name": get_clean_name(img_record), 
            "resolved_path": str(img_path), 
            "video_id": infer_video_id(get_clean_name(img_record)), 
            "source_video_id": extract_source_video(get_clean_name(img_record)), 
            "camera": extract_camera(get_clean_name(img_record)), 
            "n_annotations": len(anns_by_image.get(img_record["id"], [])) 
        })
    except Exception as e:
        unresolved_rows.append({
            "image_id": img_record["id"],
            "clean_name": get_clean_name(img_record),
            "error": repr(e)
        })

night_dataset_df = pd.DataFrame(resolved_rows) 
unresolved_df = pd.DataFrame(unresolved_rows) 

NIGHT_DATASET_CSV = CSV_DIR / "night_dataset_resolved_images.csv" 
UNRESOLVED_CSV = CSV_DIR / "night_dataset_unresolved_images.csv" 

night_dataset_df.to_csv(NIGHT_DATASET_CSV, index=False)
unresolved_df.to_csv(UNRESOLVED_CSV, index=False)

# Print
print("Resolved night dataset CSV:", NIGHT_DATASET_CSV)
print("Unresolved night dataset CSV:", UNRESOLVED_CSV)
print("Resolved images:", len(night_dataset_df))
print("Unresolved images:", len(unresolved_df))

display(night_dataset_df.head()) 

## Step 3A: Ground-truth mask decoding, union-mask building, and pixel-level metrics

In [ ]:
# Step 3A: Ground-truth mask decoding, union-mask building, and pixel-level evaluation metrics

def coco_ann_to_mask(ann, height, width):
    seg = ann.get("segmentation", None)

    if seg is None:
        return np.zeros((height, width), dtype=np.uint8)

    if isinstance(seg, list):
        if len(seg) == 0:
            return np.zeros((height, width), dtype=np.uint8)
        rles = mask_utils.frPyObjects(seg, height, width)
        rle = mask_utils.merge(rles)
        mask = mask_utils.decode(rle)

    elif isinstance(seg, dict) and "counts" in seg:
        mask = mask_utils.decode(seg)

    else:
        raise ValueError(f"Unsupported segmentation format for annotation ID {ann.get('id')}")

    if mask.ndim == 3:
        mask = np.any(mask, axis=2).astype(np.uint8)
    else:
        mask = (mask > 0).astype(np.uint8)

    return mask

# Ground truth instance masks for images
def get_gt_instance_masks_for_image(img_record, image_annotations, valid_cow_gt_category_ids=VALID_COW_GT_CATEGORY_IDS):
    height = int(img_record["height"])
    width = int(img_record["width"])

    gt_masks = []
    behavior_names = []

    for ann in image_annotations:
        cat_id = ann.get("category_id", -1)

        if cat_id in valid_cow_gt_category_ids:
            try:
                m = coco_ann_to_mask(ann, height, width)
                if m.sum() > 0:
                    gt_masks.append(m.astype(np.uint8))
                    behavior_names.append(category_id_to_name.get(cat_id, str(cat_id)))
            except Exception as e:
                print(f"[WARNING] GT mask decoding failed for annotation {ann.get('id')}: {e}")

    return gt_masks, sorted(set(behavior_names))

# Union mask
def union_masks(mask_list, shape_hw):
    union_mask = np.zeros(shape_hw, dtype=np.uint8)
    for m in mask_list:
        union_mask = np.logical_or(union_mask, m).astype(np.uint8)
    return union_mask

def safe_divide(a, b):
    return float(a) / float(b) if b != 0 else np.nan

def compute_confusion_binary(gt_mask, pred_mask):
    gt = gt_mask.astype(bool)
    pr = pred_mask.astype(bool)

    tp = np.logical_and(gt, pr).sum()
    fp = np.logical_and(~gt, pr).sum()
    fn = np.logical_and(gt, ~pr).sum()
    tn = np.logical_and(~gt, ~pr).sum()

    return tp, fp, fn, tn

# Pixel metrics computation
def compute_pixel_metrics(gt_mask, pred_mask):
    tp, fp, fn, tn = compute_confusion_binary(gt_mask, pred_mask)

    iou = safe_divide(tp, tp + fp + fn) 
    dice = safe_divide(2 * tp, 2 * tp + fp + fn) 
    precision = safe_divide(tp, tp + fp) 
    recall = safe_divide(tp, tp + fn) 
    specificity = safe_divide(tn, tn + fp) 
    accuracy = safe_divide(tp + tn, tp + tn + fp + fn) 
    balanced_accuracy = np.nanmean([recall, specificity]) 

    return {
        "tp_pixels": tp, 
        "fp_pixels": fp, 
        "fn_pixels": fn, 
        "tn_pixels": tn, 
        "iou": iou, 
        "dice": dice,
        "mask_precision": precision, 
        "mask_recall": recall,
        "mask_f1": dice, 
        "specificity": specificity, 
        "accuracy": accuracy, 
        "balanced_accuracy": balanced_accuracy, 
    }

def binary_mask_iou(mask_a, mask_b):
    a = mask_a.astype(bool)
    b = mask_b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return safe_divide(inter, union)

def binary_mask_dice(mask_a, mask_b):
    a = mask_a.astype(bool)
    b = mask_b.astype(bool)
    inter = np.logical_and(a, b).sum()
    return safe_divide(2 * inter, a.sum() + b.sum())

## Step 3B: Instance matching and threshold-based evaluation utilities

In [ ]:
# Step 3B: Hungarian instance matching and internal threshold-based evaluation functions
def hungarian_instance_matching(gt_masks, pred_masks, iou_threshold=0.50):
    n_gt = len(gt_masks)
    n_pred = len(pred_masks)

    if n_gt == 0 and n_pred == 0:
        return {
            "matches": [],
            "tp": 0, "fp": 0, "fn": 0,
            "precision": np.nan, "recall": np.nan, "f1": np.nan,
            "matched_mean_iou": np.nan, "matched_mean_dice": np.nan
        }

    if n_gt == 0:
        return {
            "matches": [],
            "tp": 0, "fp": n_pred, "fn": 0,
            "precision": 0.0, "recall": np.nan, "f1": np.nan,
            "matched_mean_iou": np.nan, "matched_mean_dice": np.nan
        }

    if n_pred == 0:
        return {
            "matches": [],
            "tp": 0, "fp": 0, "fn": n_gt,
            "precision": np.nan, "recall": 0.0, "f1": np.nan,
            "matched_mean_iou": np.nan, "matched_mean_dice": np.nan
        }

    iou_matrix = np.zeros((n_gt, n_pred), dtype=np.float32)
    dice_matrix = np.zeros((n_gt, n_pred), dtype=np.float32)

    for i, gt_m in enumerate(gt_masks):
        for j, pr_m in enumerate(pred_masks):
            iou_matrix[i, j] = binary_mask_iou(gt_m, pr_m)
            dice_matrix[i, j] = binary_mask_dice(gt_m, pr_m)

    cost_matrix = 1.0 - iou_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches = []
    matched_ious = []
    matched_dices = []

    for r, c in zip(row_ind, col_ind):
        iou_val = iou_matrix[r, c]
        dice_val = dice_matrix[r, c]
        if iou_val >= iou_threshold:
            matches.append((r, c))
            matched_ious.append(float(iou_val))
            matched_dices.append(float(dice_val))

    tp = len(matches)
    fp = n_pred - tp
    fn = n_gt - tp

    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    f1 = safe_divide(2 * precision * recall, precision + recall) if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0 else np.nan

    return {
        "matches": matches,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matched_mean_iou": float(np.mean(matched_ious)) if len(matched_ious) > 0 else np.nan,
        "matched_mean_dice": float(np.mean(matched_dices)) if len(matched_dices) > 0 else np.nan,
    }

def compute_precision_at_thresholds(gt_masks, pred_masks, thresholds=IOU_ANALYSIS_THRESHOLDS):
    rows = []
    for thr in thresholds:
        match_result = hungarian_instance_matching(gt_masks, pred_masks, iou_threshold=float(thr))
        rows.append({
            "iou_threshold": float(thr),
            "precision": match_result["precision"],
            "recall": match_result["recall"],
            "f1": match_result["f1"],
            "tp": match_result["tp"], 
            "fp": match_result["fp"], 
            "fn": match_result["fn"],
        })
    return pd.DataFrame(rows)

## Step 3C. Visualization and COCO export utilities

In [ ]:
# Step 3C: Overlay visualization, publication style, and COCO result export helpers

COLORS = {
    "overlap": "#1f77b4",  
    "pixel": "#2ca02c", 
    "instance": "#d62728", 
    "ci_mean": "#1f77b4", 
    "ci_error": "#d62728", 
}

plt.rcParams.update({
    "figure.facecolor": "white", 
    "axes.facecolor": "white", 
    "savefig.facecolor": "white", 
    "axes.edgecolor": "black", 
    "axes.linewidth": 1.2, 
    "axes.labelsize": 13, 
    "xtick.labelsize": 11, 
    "ytick.labelsize": 11, 
    "legend.fontsize": 11, 
    "font.size": 12, 
})

def axes(ax, xlabel=None, ylabel=None, use_grid=True):
    if xlabel is not None:
        ax.set_xlabel(xlabel) 
    if ylabel is not None:
        ax.set_ylabel(ylabel) 

    ax.spines["top"].set_visible(False) 
    ax.spines["right"].set_visible(False) 
    ax.spines["left"].set_linewidth(1.2) 
    ax.spines["bottom"].set_linewidth(1.2) 

    ax.tick_params(axis="both", which="major", direction="out", length=6, width=1.0) 
    ax.tick_params(axis="both", which="minor", direction="out", length=3, width=0.8)

    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())

    if use_grid:
        ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.6, alpha=0.35) 

# Defining a function to save the figure
def save_publication_figure(fig, out_base_path):
    fig.savefig(str(out_base_path) + ".png", dpi=600, bbox_inches="tight", facecolor="white") 
    fig.savefig(str(out_base_path) + ".jpg", dpi=600, bbox_inches="tight", facecolor="white")

# Overlay images
def create_overlay_image(image_bgr, gt_union_mask, pred_union_mask):
    vis = image_bgr.copy()

    green = np.zeros_like(vis)
    green[..., 1] = 255

    red = np.zeros_like(vis)
    red[..., 2] = 255

    gt_alpha = 0.25
    pred_alpha = 0.25

    if gt_union_mask is not None:
        vis = np.where(gt_union_mask[..., None].astype(bool), cv2.addWeighted(vis, 1 - gt_alpha, green, gt_alpha, 0), vis)
    if pred_union_mask is not None:
        vis = np.where(pred_union_mask[..., None].astype(bool), cv2.addWeighted(vis, 1 - pred_alpha, red, pred_alpha, 0), vis)

    return vis.astype(np.uint8)

def encode_binary_mask_to_coco_rle(mask_uint8):
    mask_fortran = np.asfortranarray(mask_uint8.astype(np.uint8))
    rle = mask_utils.encode(mask_fortran)
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle

## Step 4A. Build SAM2 image predictor robustly

In [ ]:
# Step 4A: Robust construction of SAM2 image predictor using the installed SAM2 package

def build_sam2_image_predictor(checkpoint_path, device="cuda"):
    attempted_errors = []

    try:
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        from sam2.build_sam import build_sam2
        print("SAM2 package imported successfully")
    except Exception as e:
        raise RuntimeError(
            "SAM2 package import failed. Please confirm that SAM2 is installed correctly.\n"
            f"Import error: {repr(e)}"
        )

    cfg_candidates = [
        "configs/sam2/sam2_hiera_s.yaml",
        "configs/sam2/sam2_hiera_t.yaml",
        "configs/sam2/sam2_hiera_l.yaml",
        "configs/sam2/sam2_hiera_b+.yaml",
    ]

    for cfg_name in cfg_candidates:
        try:
            print(f"Trying SAM2 config: {cfg_name}")
            try:
                sam2_model = build_sam2(
                    config_file=cfg_name,
                    ckpt_path=str(checkpoint_path),
                    device=device,
                    apply_postprocessing=False
                )
            except TypeError:
                sam2_model = build_sam2(
                    config_file=cfg_name,
                    ckpt_path=str(checkpoint_path),
                    device=device
                )

            predictor = SAM2ImagePredictor(sam2_model)
            print(f"SAM2 predictor built with config: {cfg_name}")
            return predictor, cfg_name

        except Exception as e:
            attempted_errors.append(f"{cfg_name} -> {repr(e)}")

    raise RuntimeError(
        "Could not build SAM2 image predictor.\n\n"
        + "\n".join(attempted_errors)
    )

## Step 4B. Load the detector and SAM2 models

In [ ]:
# Step 4B: Loading the detector and SAM2 model

if not DETECTOR_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Detector weights not found: {DETECTOR_WEIGHTS_PATH}")

print("Loading detector:", DETECTOR_NAME)
detector = YOLO(str(DETECTOR_WEIGHTS_PATH))

# Explicitly move YOLO to CUDA if available
if torch.cuda.is_available():
    try:
        detector.model.to("cuda")
        print("YOLO detector moved to CUDA.")
    except Exception as e:
        print(f"Could not explicitly move YOLO detector to CUDA: {e}")

# Trying detector in evaluation mode
try:
    detector.model.eval()
    print("YOLO detector set to evaluation mode.")
except Exception as e:
    print(f"[WARNING] Could not set YOLO detector to eval mode: {e}")

print("Building SAM2 predictor....")
sam2_predictor, sam2_cfg_used = build_sam2_image_predictor(
    checkpoint_path=SAM2_CHECKPOINT_PATH,
    device=SAM2_DEVICE
)
# Printing
print("SAM2 predictor ready")
print("SAM2 config used:", sam2_cfg_used)
print("SAM2 device:", SAM2_DEVICE)

try:
    if hasattr(sam2_predictor, "model"):
        sam2_predictor.model = sam2_predictor.model.to(SAM2_DEVICE)
        print(f"SAM2 model moved to {SAM2_DEVICE} explicitly")
except Exception as e:
    print(f"Could not explicitly move SAM2 model to {SAM2_DEVICE}: {e}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache has been cleared.")

## Step 4C. Detector and SAM2 inference helper functions

In [ ]:
# Step 4C: Detector and SAM2 inference helper functions
def run_detector_on_image(detector, image_bgr):
    with torch.inference_mode():
        results = detector.predict(
            source=image_bgr, 
            conf=CONF_THRESHOLD, 
            iou=IOU_THRESHOLD, 
            imgsz=IMAGE_SIZE_FOR_DETECTOR, 
            device=YOLO_DEVICE, 
            verbose=False,
            max_det=MAX_DETECTIONS_PER_IMAGE
        )

    if len(results) == 0:
        return np.empty((0, 4), dtype=np.float32), np.array([], dtype=float), np.array([], dtype=int)

    result = results[0]
    if result.boxes is None or len(result.boxes) == 0:
        return np.empty((0, 4), dtype=np.float32), np.array([], dtype=float), np.array([], dtype=int)

    boxes_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    scores = result.boxes.conf.detach().cpu().numpy().astype(float) if result.boxes.conf is not None else np.ones(len(boxes_xyxy), dtype=float)
    classes = result.boxes.cls.detach().cpu().numpy().astype(int) if result.boxes.cls is not None else np.zeros(len(boxes_xyxy), dtype=int)

    return boxes_xyxy, scores, classes

def normalize_sam_mask_output(masks):
    if masks is None:
        return None

    arr = np.asarray(masks)

    if arr.ndim == 4:
        arr = arr[0]
    if arr.ndim == 3:
        arr = arr[0]

    return arr.astype(bool)

def run_sam2_with_boxes(predictor, image_bgr, boxes_xyxy):
    if len(boxes_xyxy) == 0:
        return []

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)

    pred_masks = []
    use_amp = torch.cuda.is_available()

    for box in boxes_xyxy:
        box_np = np.array(box, dtype=np.float32)

        try:
            with torch.inference_mode():
                if use_amp:
                    with torch.amp.autocast(device_type="cuda"):
                        try:
                            masks, scores, logits = predictor.predict(
                                point_coords=None,
                                point_labels=None,
                                box=box_np[None, :],
                                multimask_output=False
                            )
                        except TypeError:
                            masks, scores, logits = predictor.predict(
                                box=box_np[None, :],
                                multimask_output=False
                            )
                else:
                    try:
                        masks, scores, logits = predictor.predict(
                            point_coords=None,
                            point_labels=None,
                            box=box_np[None, :],
                            multimask_output=False
                        )
                    except TypeError:
                        masks, scores, logits = predictor.predict(
                            box=box_np[None, :],
                            multimask_output=False
                        )
        except Exception as e:
            print(f"SAM2 prediction failed for box {box_np.tolist()}: {e}")
            continue

        mask_bool = normalize_sam_mask_output(masks)
        if mask_bool is None:
            continue

        if mask_bool.max() == 0:
            continue

        pred_masks.append(mask_bool.astype(np.uint8))

    return pred_masks

## Diagnostic cell: GPU Verification 

In [ ]:
# GPU Diagnostic and verification 

print("\nTorch CUDA status:") 
print("CUDA available:", torch.cuda.is_available()) 

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device()) 
    print("Device name:", torch.cuda.get_device_name(0)) 
    print("CUDA version:", torch.version.cuda) 
    print("cuDNN enabled:", torch.backends.cudnn.enabled) 
else:
    print("CUDA is not available.")

## Step 5A. Main scientific evaluation loop

In [ ]:
# Step 5A: Main scientific evaluation loop
sample_rows = []
sample_class_counter = Counter()
sample_audit_errors = []
sample_n = min(20, len(night_images))

for img_record in night_images[:sample_n]:
    try:
        img_path = resolve_image_path(img_record)
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise RuntimeError(f"OpenCV could not read image: {img_path}")

        boxes_xyxy, confs, classes = run_detector_on_image(detector, image_bgr)

        for c in classes:
            sample_class_counter[int(c)] += 1

        sample_rows.append({
            "image_id": img_record["id"], 
            "file_name": get_clean_name(img_record), 
            "n_boxes": len(boxes_xyxy), 
            "mean_conf": float(np.mean(confs)) if len(confs) > 0 else np.nan, 
            "classes": classes.tolist() if len(classes) > 0 else [] 
        })
    except Exception as e:
        sample_audit_errors.append({
            "image_id": img_record.get("id"),
            "file_name": get_clean_name(img_record),
            "error": repr(e)
        })

sample_audit_df = pd.DataFrame(sample_rows) 
sample_audit_error_df = pd.DataFrame(sample_audit_errors)
print("Detector sample audit completed.") 
print("Predicted class distribution in sample images:")
print(sample_class_counter)
if len(sample_audit_error_df) > 0:
    print("Sample audit images with errors:", len(sample_audit_error_df))
display(sample_audit_df.head(10)) 

per_image_rows = [] 
precision_rows = [] 
error_rows = [] 
coco_predictions = [] 

save_overlay_counter = 0
start_time = time.time()

for idx, img_record in enumerate(tqdm(night_images, desc="Evaluating NIGHT images", unit="image"), start=1):
    image_id = img_record["id"]
    file_name = get_clean_name(img_record)

    try:
        img_path = resolve_image_path(img_record)
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise RuntimeError(f"OpenCV could not read image: {img_path}")

        height, width = image_bgr.shape[:2]
        image_annotations = anns_by_image[image_id]

        camera = extract_camera(file_name)
        video_id = infer_video_id(file_name)
        source_video_id = extract_source_video(file_name)

        # Ground truth [GT] construction
        gt_masks, behavior_names = get_gt_instance_masks_for_image(
            {"id": image_id, "height": height, "width": width},
            image_annotations,
            valid_cow_gt_category_ids=VALID_COW_GT_CATEGORY_IDS
        )
        gt_union = union_masks(gt_masks, (height, width))

        # Detector inference
        boxes_xyxy, confs, classes = run_detector_on_image(detector, image_bgr)

        boxes_xyxy = np.asarray(boxes_xyxy, dtype=np.float32)
        confs = np.asarray(confs, dtype=float) if len(confs) > 0 else np.array([], dtype=float)
        classes = np.asarray(classes, dtype=int) if len(classes) > 0 else np.array([], dtype=int)

        if USE_CLASS_FILTER and TARGET_CLASS_ID is not None and len(classes) > 0:
            keep_idx = np.where(classes == TARGET_CLASS_ID)[0]
            boxes_xyxy = boxes_xyxy[keep_idx]
            confs = confs[keep_idx] if len(confs) == len(classes) else np.ones(len(keep_idx), dtype=float)
            classes = classes[keep_idx]

        # SAM2 segmentation
        pred_masks = run_sam2_with_boxes(sam2_predictor, image_bgr, boxes_xyxy)
        pred_union = union_masks(pred_masks, (height, width))

        # Pixel-level metrics
        pixel_metrics = compute_pixel_metrics(gt_union, pred_union)
        difficulty_bucket = infer_difficulty_bucket(len(gt_masks))

        # Instance-level matching
        match_result = hungarian_instance_matching(
            gt_masks=gt_masks,
            pred_masks=pred_masks,
            iou_threshold=PRIMARY_INSTANCE_IOU_MATCH_THRESHOLD
        )

        # Threshold-based internal precision analysis
        precision_df = compute_precision_at_thresholds(gt_masks, pred_masks, thresholds=IOU_ANALYSIS_THRESHOLDS) # Precision dataset
        precision_at_50 = precision_df.loc[np.isclose(precision_df["iou_threshold"], 0.50), "precision"].mean() 
        precision_at_75 = precision_df.loc[np.isclose(precision_df["iou_threshold"], 0.75), "precision"].mean()
        mean_precision_50_95 = precision_df["precision"].mean()

        for _, r in precision_df.iterrows():
            precision_rows.append({
                "image_id": image_id,
                "file_name": file_name, 
                "camera": camera, 
                "video_id": video_id, 
                "difficulty_bucket": difficulty_bucket, 
                **r.to_dict()
            })

        # Per-image result row
        per_image_rows.append({
            "image_id": image_id, 
            "file_name": file_name, 
            "image_path": str(img_path), 
            "height": height, 
            "width": width, 
            "camera": camera, 
            "video_id": video_id, 
            "behavior_names": behavior_names, 
            "difficulty_bucket": difficulty_bucket,

            "gt_instance_count": len(gt_masks), 
            "pred_box_count": int(len(boxes_xyxy)), 
            "pred_mask_count": int(len(pred_masks)), 
            "avg_detector_confidence": float(np.mean(confs)) if len(confs) > 0 else np.nan, 
            "min_detector_confidence": float(np.min(confs)) if len(confs) > 0 else np.nan, 
            "max_detector_confidence": float(np.max(confs)) if len(confs) > 0 else np.nan, 

            **pixel_metrics,

            "inst_tp": match_result["tp"], 
            "inst_fp": match_result["fp"], 
            "inst_fn": match_result["fn"], 
            "inst_precision": match_result["precision"], 
            "inst_recall": match_result["recall"], 
            "inst_f1": match_result["f1"], 
            "matched_mean_iou": match_result["matched_mean_iou"], 
            "matched_mean_dice": match_result["matched_mean_dice"], 

            "precision_at_iou_0.50": precision_at_50, 
            "precision_at_iou_0.75": precision_at_75, 
            "mean_precision_across_iou_0.50_0.95": mean_precision_50_95, 
        })

        # Prediction export
        pred_score_list = confs.tolist() if len(confs) == len(pred_masks) else [1.0] * len(pred_masks)

        for pred_mask, score in zip(pred_masks, pred_score_list):
            if pred_mask.sum() == 0:
                continue
            rle = encode_binary_mask_to_coco_rle(pred_mask.astype(np.uint8))
            coco_predictions.append({
                "image_id": int(image_id), 
                "category_id": int(COCO_EVAL_CATEGORY_ID), 
                "segmentation": rle, 
                "score": float(score) 
            })

        # Periodic cache cleanup
        if torch.cuda.is_available() and idx % 25 == 0:
            torch.cuda.empty_cache()

    except Exception as e:
        error_rows.append({
            "image_id": image_id, 
            "file_name": file_name, 
            "error": repr(e), 
            "traceback": traceback.format_exc() 
        })

elapsed_min = (time.time() - start_time) / 60.0
print(f"Main evaluation loop have been completed in {elapsed_min:.2f} minutes")
print("Successful images:", len(per_image_rows))
print("Failed images:", len(error_rows))
print("COCO-format predictions:", len(coco_predictions))

## Step 5B. Save per-image outputs and error logs

In [ ]:
# Step 5B: Save per-image results, threshold analysis, and error reports
per_image_df = pd.DataFrame(per_image_rows)
precision_df_all = pd.DataFrame(precision_rows)
error_df = pd.DataFrame(error_rows)

PER_IMAGE_CSV = CSV_DIR / "night_yolov11_sam2_per_image_metrics.csv" 
PRECISION_CSV = CSV_DIR / "night_yolov11_sam2_internal_threshold_metrics.csv" 
ERROR_CSV = CSV_DIR / "night_yolov11_sam2_errors.csv" 

per_image_df.to_csv(PER_IMAGE_CSV, index=False) 
precision_df_all.to_csv(PRECISION_CSV, index=False) 
error_df.to_csv(ERROR_CSV, index=False) 

# Print
print("Saved per-image metrics:", PER_IMAGE_CSV)
print("Saved internal threshold metrics:", PRECISION_CSV)
print("Saved error log:", ERROR_CSV)

print("\nCore evaluation sanity check")
print("Rows in per-image metrics:", len(per_image_df))
print("Mean ground truth [GT] instance count:", per_image_df["gt_instance_count"].mean() if len(per_image_df) else np.nan)
print("Mean predicted box count:", per_image_df["pred_box_count"].mean() if len(per_image_df) else np.nan)
print("Mean predicted mask count:", per_image_df["pred_mask_count"].mean() if len(per_image_df) else np.nan)
print("Mean IoU:", per_image_df["iou"].mean() if len(per_image_df) else np.nan)
print("Mean Dice:", per_image_df["dice"].mean() if len(per_image_df) else np.nan)
print("Mean Instance F1:", per_image_df["inst_f1"].mean() if len(per_image_df) else np.nan)

display(per_image_df.head())

## Step 5C: Summary tables

In [ ]:
# Step 5C: Summary table
if len(per_image_df) == 0:
    raise ValueError("No valid evaluation rows were created.")

# Evaluation metrics columns
metric_cols = [
    "iou", "dice", "mask_precision", "mask_recall", "mask_f1", 
    "specificity", "accuracy", "balanced_accuracy", 
    "inst_precision", "inst_recall", "inst_f1", 
    "matched_mean_iou", "matched_mean_dice", 
    "precision_at_iou_0.50", "precision_at_iou_0.75", "mean_precision_across_iou_0.50_0.95" 
]

count_cols = [
    "gt_instance_count", "pred_box_count", "pred_mask_count", 
    "inst_tp", "inst_fp", "inst_fn", 
    "tp_pixels", "fp_pixels", "fn_pixels", "tn_pixels" 
]

# Defining a function for the summary dataframe
def summarize_df(df, group_type, group_name):
    row = {"group_type": group_type, "group_name": group_name, "n_images": int(len(df))}
    for c in metric_cols + count_cols:
        row[c] = pd.to_numeric(df[c], errors="coerce").mean()
    return row

summary_rows = [summarize_df(per_image_df, "overall", "overall")] 

for bucket, sub_df in per_image_df.groupby("difficulty_bucket", dropna=False):
    summary_rows.append(summarize_df(sub_df, "difficulty_bucket", bucket))

for video_id, sub_df in per_image_df.groupby("video_id", dropna=False):
    summary_rows.append(summarize_df(sub_df, "video_id", video_id))

summary_df = pd.DataFrame(summary_rows)  

SUMMARY_CSV = CSV_DIR / "night_yolov11_sam2_summary.csv" 
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved summary:", SUMMARY_CSV)
display(summary_df.head(20)) 

## Step 5D. Confidence intervals and error-category analysis

In [ ]:
# Step 5D: Confidence intervals, error categories, and worst-case scientific analysis

SEED = 42

def bootstrap_ci(series, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    values = pd.to_numeric(series, errors="coerce").dropna().values

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means.append(np.mean(sample))

    mean_val = np.mean(values)
    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return mean_val, lower, upper

# Confidence interval evaluation metrics
ci_metrics = [
    "iou", "dice", "mask_precision", "mask_recall", "mask_f1", 
    "inst_f1", "precision_at_iou_0.50", "precision_at_iou_0.75", "mean_precision_across_iou_0.50_0.95"
]

# Confidence interval rows
ci_rows = []
for metric in ci_metrics:
    mean_val, lower, upper = bootstrap_ci(per_image_df[metric], n_boot=2000, alpha=0.05, seed=SEED)
    ci_rows.append({ 
        "metric": metric, 
        "mean": mean_val, 
        "ci95_lower": lower, 
        "ci95_upper": upper 
    })

ci_df = pd.DataFrame(ci_rows) 
CI_CSV = CSV_DIR / "night_yolov11_sam2_confidence_intervals.csv" 
ci_df.to_csv(CI_CSV, index=False)

print("Saved confidence intervals:", CI_CSV)
display(ci_df) 

# Defining a function for error category
def assign_error_category(row):
    gt_n = row["gt_instance_count"]
    pred_n = row["pred_mask_count"]
    iou = row["iou"]
    inst_f1 = row["inst_f1"]

    if gt_n > 0 and pred_n == 0:
        return "complete_miss"
    if gt_n == 0 and pred_n > 0:
        return "false_positive_image"
    if pred_n < gt_n:
        return "under_segmentation_or_missed_instances"
    if pred_n > gt_n:
        return "over_segmentation_or_extra_instances"
    if pd.notna(iou) and iou < 0.30:
        return "poor_mask_overlap"
    if pd.notna(inst_f1) and inst_f1 < 0.30:
        return "poor_instance_separation"
    return "acceptable_or_mixed"

per_image_df["error_category"] = per_image_df.apply(assign_error_category, axis=1)

ERROR_CATEGORY_CSV = CSV_DIR / "night_yolov11_sam2_error_categories.csv" 
per_image_df.to_csv(ERROR_CATEGORY_CSV, index=False)

print("Saved per-image metrics with error categories:", ERROR_CATEGORY_CSV)

worst_iou_df = per_image_df.sort_values("iou", ascending=True).head(SAVE_WORST_CASE_OVERLAYS).copy() 
worst_instf1_df = per_image_df.sort_values("inst_f1", ascending=True).head(SAVE_WORST_CASE_OVERLAYS).copy()  

WORST_IOU_CSV = CSV_DIR / "night_yolov11_sam2_worst_iou_cases.csv" 
WORST_INSTF1_CSV = CSV_DIR / "night_yolov11_sam2_worst_instf1_cases.csv" 

worst_iou_df.to_csv(WORST_IOU_CSV, index=False) 
worst_instf1_df.to_csv(WORST_INSTF1_CSV, index=False) 

print("Saved worst-IoU cases:", WORST_IOU_CSV)
print("Saved worst-instance-F1 cases:", WORST_INSTF1_CSV)

## Step 5E: Worst-case overlays

In [ ]:
# Step 5E: Exporting worst-case overlays
worst_case_names = set(worst_iou_df["file_name"].tolist()) | set(worst_instf1_df["file_name"].tolist())

for img_record in tqdm(night_images, desc="Saving worst-case overlays", unit="image"):
    file_name = get_clean_name(img_record)
    if file_name not in worst_case_names:
        continue

    try:
        img_path = resolve_image_path(img_record)
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            continue

        height, width = image_bgr.shape[:2]
        image_annotations = anns_by_image[img_record["id"]]

        gt_masks, _ = get_gt_instance_masks_for_image(
            {"id": img_record["id"], "height": height, "width": width},
            image_annotations,
            valid_cow_gt_category_ids=VALID_COW_GT_CATEGORY_IDS
        )
        gt_union = union_masks(gt_masks, (height, width))

        boxes_xyxy, confs, classes = run_detector_on_image(detector, image_bgr)

        boxes_xyxy = np.asarray(boxes_xyxy, dtype=np.float32)
        if USE_CLASS_FILTER and TARGET_CLASS_ID is not None and len(classes) > 0:
            keep_idx = np.where(classes == TARGET_CLASS_ID)[0]
            boxes_xyxy = boxes_xyxy[keep_idx]

        pred_masks = run_sam2_with_boxes(sam2_predictor, image_bgr, boxes_xyxy)
        pred_union = union_masks(pred_masks, (height, width))

        overlay = create_overlay_image(image_bgr, gt_union, pred_union)
        out_path = WORST_CASE_DIR / f"{Path(file_name).stem}_worst_overlay.jpg"
        cv2.imwrite(str(out_path), overlay)

    except Exception as e:
        print(f"[WARNING] Could not save worst-case overlay for {file_name}: {e}")

print("Worst-case overlay export completed")

## Step 6A: Standard COCO segmentation evaluation

In [ ]:
# Step 6A: Standard COCO segmentation evaluation
night_image_ids = {img["id"] for img in night_images}

class_agnostic_images = [
    copy.deepcopy(img) for img in images
    if img["id"] in night_image_ids
]

class_agnostic_annotations = []
for ann in annotations:
    if ann["image_id"] not in night_image_ids:
        continue
    if ann.get("category_id") not in VALID_COW_GT_CATEGORY_IDS:
        continue
    if ann.get("segmentation") in [None, [], {}]:
        continue

    ann_copy = copy.deepcopy(ann)
    ann_copy["category_id"] = int(COCO_EVAL_CATEGORY_ID)
    ann_copy["iscrowd"] = int(ann_copy.get("iscrowd", 0))
    class_agnostic_annotations.append(ann_copy)

class_agnostic_coco = {
    "info": copy.deepcopy(coco_json.get("info", {})),
    "licenses": copy.deepcopy(coco_json.get("licenses", [])),
    "images": class_agnostic_images,
    "annotations": class_agnostic_annotations,
    "categories": [
        {"id": int(COCO_EVAL_CATEGORY_ID), "name": "cow", "supercategory": "cow"}
    ],
}

CLASS_AGNOSTIC_GT_JSON = JSON_DIR / "night_class_agnostic_coco_gt.json"  
CLASS_AGNOSTIC_PRED_JSON = JSON_DIR / "night_class_agnostic_coco_predictions.json"  

with open(CLASS_AGNOSTIC_GT_JSON, "w", encoding="utf-8") as f:
    json.dump(class_agnostic_coco, f)

with open(CLASS_AGNOSTIC_PRED_JSON, "w", encoding="utf-8") as f:
    json.dump(coco_predictions, f)

coco_metric_rows = []

if len(class_agnostic_annotations) == 0:
    print("No class-agnostic GT annotations were available for standard COCO segmentation evaluation.")
    coco_summary_df = pd.DataFrame(columns=["metric", "value"])
elif len(coco_predictions) == 0:
    print("No predictions were available for standard COCO segmentation evaluation.")
    coco_summary_df = pd.DataFrame(columns=["metric", "value"])
else:
    coco_gt = COCO(str(CLASS_AGNOSTIC_GT_JSON))
    coco_dt = coco_gt.loadRes(str(CLASS_AGNOSTIC_PRED_JSON))

    coco_eval = COCOeval(coco_gt, coco_dt, iouType="segm")
    coco_eval.params.imgIds = sorted(list(night_image_ids))
    coco_eval.params.catIds = [int(COCO_EVAL_CATEGORY_ID)]
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    metric_names = [
        "AP@[0.50:0.95]",
        "AP@0.50",
        "AP@0.75",
        "AP_small",
        "AP_medium",
        "AP_large",
        "AR@1",
        "AR@10",
        "AR@100",
        "AR_small",
        "AR_medium",
        "AR_large",
    ]

    for metric_name, metric_value in zip(metric_names, coco_eval.stats.tolist()):
        coco_metric_rows.append({
            "metric": metric_name,
            "value": float(metric_value)
        })

    coco_summary_df = pd.DataFrame(coco_metric_rows)

COCO_SUMMARY_CSV = CSV_DIR / "night_yolov11_sam2_coco_summary.csv" 
coco_summary_df.to_csv(COCO_SUMMARY_CSV, index=False)

print("Temporary class-agnostic GT JSON:", CLASS_AGNOSTIC_GT_JSON)
print("Temporary class-agnostic prediction JSON:", CLASS_AGNOSTIC_PRED_JSON)
print("COCO summary CSV:", COCO_SUMMARY_CSV)

display(coco_summary_df) 

## Step 7A: Aggregation and stratification

In [ ]:
# Step 7A: Camera/source-wise, behavior-wise, and camera-by-difficulty stratification and aggregation

camera_summary_df = (
    per_image_df
    .groupby("camera", dropna=False)[ 
        ["iou", "dice", "mask_precision", "mask_recall", "inst_f1", 
         "precision_at_iou_0.50", "precision_at_iou_0.75", "mean_precision_across_iou_0.50_0.95"] 
    ]
    .mean()
    .reset_index()
)

CAMERA_SUMMARY_CSV = CSV_DIR / "night_yolov11_sam2_camera_summary.csv" 
camera_summary_df.to_csv(CAMERA_SUMMARY_CSV, index=False)

print("Camera/source-wise summary saved:", CAMERA_SUMMARY_CSV)
display(camera_summary_df)

if "source_video_id" in per_image_df.columns:
    source_video_summary_df = (
        per_image_df
        .groupby("source_video_id", dropna=False)[
            ["iou", "dice", "mask_precision", "mask_recall", "inst_f1",
             "precision_at_iou_0.50", "precision_at_iou_0.75", "mean_precision_across_iou_0.50_0.95"]
        ]
        .mean()
        .reset_index()
    )
else:
    source_video_summary_df = pd.DataFrame()

SOURCE_VIDEO_SUMMARY_CSV = CSV_DIR / f"{EVAL_SPLIT_NAME}_yolov11_sam2_source_video_summary.csv"
source_video_summary_df.to_csv(SOURCE_VIDEO_SUMMARY_CSV, index=False)
print("Source-video summary saved:", SOURCE_VIDEO_SUMMARY_CSV)
display(source_video_summary_df.head())

behavior_rows = []
for _, row in per_image_df.iterrows():
    behavior_names = row["behavior_names"]

    if isinstance(behavior_names, str):
        try:
            behavior_names = ast.literal_eval(behavior_names)
        except Exception:
            behavior_names = [behavior_names]

    if not isinstance(behavior_names, list):
        behavior_names = [behavior_names]
    # For Loop
    for behavior in behavior_names:
        behavior_rows.append({
            "behavior": behavior, 
            "iou": row["iou"], 
            "dice": row["dice"], 
            "mask_precision": row["mask_precision"], 
            "mask_recall": row["mask_recall"], 
            "inst_f1": row["inst_f1"], 
            "precision_at_iou_0.50": row["precision_at_iou_0.50"], 
            "precision_at_iou_0.75": row["precision_at_iou_0.75"], 
            "mean_precision_across_iou_0.50_0.95": row["mean_precision_across_iou_0.50_0.95"], 
        })

behavior_df = pd.DataFrame(behavior_rows) 

if len(behavior_df) > 0:
    behavior_summary_df = (
        behavior_df
        .groupby("behavior", dropna=False)[ 
            ["iou", "dice", "mask_precision", "mask_recall", "inst_f1",  
             "precision_at_iou_0.50", "precision_at_iou_0.75", "mean_precision_across_iou_0.50_0.95"] 
        ]
        .mean() 
        .reset_index() 
    )
else:
    behavior_summary_df = pd.DataFrame() 

BEHAVIOR_SUMMARY_CSV = CSV_DIR / "night_yolov11_sam2_behavior_summary.csv" 
behavior_summary_df.to_csv(BEHAVIOR_SUMMARY_CSV, index=False)

# Print
print("Behavior-wise summary saved:", BEHAVIOR_SUMMARY_CSV)
display(behavior_summary_df.head() if len(behavior_summary_df) > 0 else behavior_summary_df)


camera_difficulty_summary_df = (
    per_image_df
    .groupby(["camera", "difficulty_bucket"], dropna=False)[ 
        ["iou", "dice", "inst_f1", "mean_precision_across_iou_0.50_0.95"] 
    ]
    .mean()
    .reset_index() 
)

CAMERA_DIFFICULTY_SUMMARY_CSV = CSV_DIR / "night_yolov11_sam2_camera_difficulty_summary.csv" 
camera_difficulty_summary_df.to_csv(CAMERA_DIFFICULTY_SUMMARY_CSV, index=False) 

print("Camera × difficulty summary saved:", CAMERA_DIFFICULTY_SUMMARY_CSV)
display(camera_difficulty_summary_df) 

## Step 7B: Visualization

In [ ]:
# Step 7B: Visualization
# A. Primary metrics for figure
primary_metric_specs = [
    {"col": "iou", "label": "IoU", "group": "Overlap metrics", "color": COLORS["overlap"]},
    {"col": "dice", "label": "Dice", "group": "Overlap metrics", "color": COLORS["overlap"]},
    {"col": "mask_precision", "label": "Pixel precision", "group": "Pixel-level metrics", "color": COLORS["pixel"]},
    {"col": "mask_recall", "label": "Pixel recall", "group": "Pixel-level metrics", "color": COLORS["pixel"]},
    {"col": "mask_f1", "label": "Pixel F1", "group": "Pixel-level metrics", "color": COLORS["pixel"]},
    {"col": "inst_f1", "label": "Instance F1", "group": "Instance-level metrics", "color": COLORS["instance"]},
    {"col": "precision_at_iou_0.50", "label": "P@IoU 0.50", "group": "Instance-level metrics", "color": COLORS["instance"]},
    {"col": "precision_at_iou_0.75", "label": "P@IoU 0.75", "group": "Instance-level metrics", "color": COLORS["instance"]},
    {"col": "mean_precision_across_iou_0.50_0.95", "label": "Mean P 0.50-0.95", "group": "Instance-level metrics", "color": COLORS["instance"]},
]

primary_labels = [m["label"] for m in primary_metric_specs]
primary_values = [pd.to_numeric(per_image_df[m["col"]], errors="coerce").mean() for m in primary_metric_specs]
primary_colors = [m["color"] for m in primary_metric_specs]

# Generating figures
fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="white") 
# Bar plot
bars = ax.bar(
    primary_labels,
    primary_values,
    color=primary_colors,
    edgecolor="black",
    linewidth=0.8 
)

axes(ax, xlabel="Evaluation metric", ylabel="Mean value", use_grid=True) 
ax.set_ylim(0, 1.05)
plt.xticks(rotation=40, ha="right")
# Figure legends
legend_handles = [
    Patch(facecolor=COLORS["overlap"], edgecolor="black", label="Overlap metrics"),
    Patch(facecolor=COLORS["pixel"], edgecolor="black", label="Pixel-level metrics"),
    Patch(facecolor=COLORS["instance"], edgecolor="black", label="Instance-level metrics"),
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False) 

plt.tight_layout()
save_publication_figure(fig, FIG_DIR / "night_yolov11_sam2_primary_metrics") 
plt.show() # display figure

# B. Difficulty-wise summary plots
difficulty_order = ["easy", "medium", "hard"] 
# Difficulty summary
difficulty_summary = (
    per_image_df.groupby("difficulty_bucket")[[
        "iou", "dice", "inst_f1", "mean_precision_across_iou_0.50_0.95"
    ]]
    .mean()
    .reset_index()
)
# Difficulty plot
difficulty_summary = (
    difficulty_summary
    .set_index("difficulty_bucket")
    .reindex(difficulty_order)
    .reset_index()
)
# Difficulty plots specification
difficulty_plot_specs = [
    ("iou", "IoU", "night_yolov11_sam2_iou_by_difficulty", COLORS["overlap"]),
    ("dice", "Dice", "night_yolov11_sam2_dice_by_difficulty", COLORS["overlap"]),
    ("inst_f1", "Instance F1", "night_yolov11_sam2_inst_f1_by_difficulty", COLORS["instance"]),
    ("mean_precision_across_iou_0.50_0.95", "Mean P 0.50-0.95", "night_yolov11_sam2_mean_precision_across_iou_0.50_0.95_by_difficulty", COLORS["instance"]),
]

for metric_col, y_label, out_name, bar_color in difficulty_plot_specs:
    fig, ax = plt.subplots(figsize=(7.5, 5.5), facecolor="white")
# bar charts
    ax.bar(
        difficulty_summary["difficulty_bucket"],
        difficulty_summary[metric_col],
        color=bar_color,
        edgecolor="black",
        linewidth=0.8
    )

    axes(ax, xlabel="Difficulty group", ylabel=y_label, use_grid=True) 
    ax.set_ylim(0, 1.05)

    plt.tight_layout() 
    save_publication_figure(fig, FIG_DIR / out_name) 
    plt.show() 

# C. Confidence interval figure
ci_metric_order = [
    "iou",
    "dice",
    "mask_precision",
    "mask_recall",
    "mask_f1",
    "inst_f1",
    "precision_at_iou_0.50",
    "precision_at_iou_0.75",
    "mean_precision_across_iou_0.50_0.95"
]

ci_label_map = {
    "iou": "IoU",
    "dice": "Dice",
    "mask_precision": "Pixel precision",
    "mask_recall": "Pixel recall",
    "mask_f1": "Pixel F1",
    "inst_f1": "Instance F1",
    "precision_at_iou_0.50": "P@IoU 0.50",
    "precision_at_iou_0.75": "P@IoU 0.75",
    "mean_precision_across_iou_0.50_0.95": "Mean P 0.50-0.95",
}

ci_plot_df = ci_df.copy()
ci_plot_df = ci_plot_df[ci_plot_df["metric"].isin(ci_metric_order)].copy()
ci_plot_df["metric"] = pd.Categorical(ci_plot_df["metric"], categories=ci_metric_order, ordered=True)
ci_plot_df = ci_plot_df.sort_values("metric").reset_index(drop=True)
ci_plot_df["metric_label"] = ci_plot_df["metric"].map(ci_label_map)

x = np.arange(len(ci_plot_df))
means = ci_plot_df["mean"].to_numpy()
lower_err = means - ci_plot_df["ci95_lower"].to_numpy()
upper_err = ci_plot_df["ci95_upper"].to_numpy() - means

fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="white")

ax.errorbar(
    x,
    means,
    yerr=[lower_err, upper_err],
    fmt="o",
    color=COLORS["ci_mean"],
    ecolor=COLORS["ci_error"],
    elinewidth=1.3,
    capsize=4,
    markersize=6,
    label="Mean ± 95% CI"
)

ax.set_xticks(x)
ax.set_xticklabels(ci_plot_df["metric_label"], rotation=40, ha="right") 
axes(ax, xlabel="Evaluation metric", ylabel="Estimated mean value", use_grid=True)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)

plt.tight_layout()
save_publication_figure(fig, FIG_DIR / "night_yolov11_sam2_confidence_intervals")
plt.show()

## Step 8A: Final summary report

In [ ]:
# Step 8A: Save final summary report
overall_rows = summary_df.loc[summary_df["group_name"] == "overall"]

if len(overall_rows) == 0:
    raise ValueError("Overall summary row not found")

overall_summary = overall_rows.iloc[0].to_dict()

coco_metric_lookup = {}
if len(coco_summary_df) > 0:
    coco_metric_lookup = dict(zip(coco_summary_df["metric"], coco_summary_df["value"]))

# Final summary report dataframe
final_report_df = pd.DataFrame([{
    "detector_name": DETECTOR_NAME, 
    "detector_weights_path": str(DETECTOR_WEIGHTS_PATH), 
    "sam2_checkpoint_path": str(SAM2_CHECKPOINT_PATH), 
    "sam2_config_used": sam2_cfg_used, 
    "evaluation_split": EVAL_SPLIT_NAME, 
    "model_name": MODEL_NAME,
    "output_root": str(EVAL_OUTPUT_ROOT),
    "difficulty_definition": "easy=1-2 cows, medium=3-5 cows, hard=6+ cows",
    "n_images": len(per_image_df), 
    "failed_images": len(error_df), 

    "mean_iou": overall_summary.get("iou", np.nan), 
    "mean_dice": overall_summary.get("dice", np.nan), 
    "mean_pixel_precision": overall_summary.get("mask_precision", np.nan), 
    "mean_pixel_recall": overall_summary.get("mask_recall", np.nan), 
    "mean_pixel_f1": overall_summary.get("mask_f1", np.nan), 
    "mean_specificity": overall_summary.get("specificity", np.nan), 
    "mean_accuracy": overall_summary.get("accuracy", np.nan), 
    "mean_balanced_accuracy": overall_summary.get("balanced_accuracy", np.nan), 

    "mean_instance_precision": overall_summary.get("inst_precision", np.nan), 
    "mean_instance_recall": overall_summary.get("inst_recall", np.nan), 
    "mean_instance_f1": overall_summary.get("inst_f1", np.nan), 
    "mean_matched_iou": overall_summary.get("matched_mean_iou", np.nan), 
    "mean_matched_dice": overall_summary.get("matched_mean_dice", np.nan), 

    "mean_precision_at_iou_0.50": overall_summary.get("precision_at_iou_0.50", np.nan), 
    "mean_precision_at_iou_0.75": overall_summary.get("precision_at_iou_0.75", np.nan), 
    "mean_precision_across_iou_0.50_0.95": overall_summary.get("mean_precision_across_iou_0.50_0.95", np.nan), 

    "coco_AP_0.50_0.95": coco_metric_lookup.get("AP@[0.50:0.95]", np.nan), 
    "coco_AP_0.50": coco_metric_lookup.get("AP@0.50", np.nan), 
    "coco_AP_0.75": coco_metric_lookup.get("AP@0.75", np.nan), 
    "coco_AR_100": coco_metric_lookup.get("AR@100", np.nan), 
}])

FINAL_REPORT_CSV = CSV_DIR / "night_yolov11_sam2_final_report.csv" 
final_report_df.to_csv(FINAL_REPORT_CSV, index=False)

# Print
print("Final report CSV saved:", FINAL_REPORT_CSV)
display(final_report_df) 

print("SEGMENTATION EVALUATION NOTEBOOK HAVE BEEN PREPARED.")
print("Detector:", DETECTOR_NAME) 
print("Detector weights:", DETECTOR_WEIGHTS_PATH) 
print("SAM2 checkpoint:", SAM2_CHECKPOINT_PATH) 
print("SAM2 config used:", sam2_cfg_used) 
print("COCO JSON:", COCO_JSON_PATH) 
print("Images evaluated:", len(per_image_df)) 
print("Images failed:", len(error_df)) 
print("Output root:", EVAL_OUTPUT_ROOT) 

print("Per-image metrics CSV:", PER_IMAGE_CSV) 
print("Internal threshold metrics CSV:", PRECISION_CSV) 
print("Summary CSV:", SUMMARY_CSV) 
print("Confidence interval CSV:", CI_CSV) 
print("Camera summary CSV:", CAMERA_SUMMARY_CSV) 
print("Behavior summary CSV:", BEHAVIOR_SUMMARY_CSV) 
print("Camera × difficulty CSV:", CAMERA_DIFFICULTY_SUMMARY_CSV) 
print("COCO summary CSV:", COCO_SUMMARY_CSV) 
print("Final report CSV:", FINAL_REPORT_CSV) 

print("Figures folder:", FIG_DIR) 
print("Worst-case overlays folder:", WORST_CASE_DIR)